# Laboratorio: cálculo de la SVD y pseudoinversa

El propósito es construir una SVD a mano con apoyo simbólico, reconocer los tamaños de sus factores y usar la pseudoinversa. Antes de ejecutar cada bloque, predice el rango, los valores singulares y las dimensiones esperadas.

## 1. Matriz de trabajo

Usaremos una matriz rectangular con valores singulares distintos:

$$
A=\begin{pmatrix}3&0\\4&0\\0&2\end{pmatrix}.
$$

In [ ]:
import sympy as sp
import numpy as np
sp.init_printing()

A = sp.Matrix([[3, 0], [4, 0], [0, 2]])
m, n = A.shape
r = A.rank()
A, (m, n, r)

## 2. Valores singulares desde la matriz asociada

Los vectores singulares derechos son vectores propios ortonormales de $A^TA$. Los valores singulares son las raíces cuadradas no negativas de sus valores propios.

In [ ]:
ATA = A.T * A
datos = ATA.eigenvects()
assert ATA == sp.diag(25, 4)
ATA, datos

## 3. Construcción de los vectores izquierdos

Ordenamos $\sigma_1=5$ y $\sigma_2=2$, tomamos $v_1=e_1$, $v_2=e_2$ y calculamos

$$
u_i=\frac{Av_i}{\sigma_i}.
$$

In [ ]:
v1 = sp.Matrix([1, 0])
v2 = sp.Matrix([0, 1])
sigma1, sigma2 = sp.Integer(5), sp.Integer(2)
u1 = A*v1/sigma1
u2 = A*v2/sigma2
assert u1.dot(u1) == 1 and u2.dot(u2) == 1
assert u1.dot(u2) == 0
assert A*v1 == sigma1*u1 and A*v2 == sigma2*u2
u1, u2

## 4. SVD completa y reducida

Para la SVD completa se añade un vector unitario $u_3$ en $\ker(A^T)$. En la SVD reducida se conservan solo las columnas asociadas a valores singulares positivos.

In [ ]:
u3 = sp.Matrix([-4, 3, 0])/5
U = sp.Matrix.hstack(u1, u2, u3)
Sigma = sp.Matrix([[5, 0], [0, 2], [0, 0]])
V = sp.eye(2)
assert U.T*U == sp.eye(3)
assert V.T*V == sp.eye(2)
assert U*Sigma*V.T == A

Ur = U[:, :2]
Sigmar = sp.diag(5, 2)
Vr = V[:, :2]
assert Ur*Sigmar*Vr.T == A
U, Sigma, V

## 5. Espacios fundamentales

Las columnas de $U_r$ generan la imagen y las de $V_r$ generan el espacio fila. Los vectores usados para completar $U$ generan el núcleo de $A^T$.

In [ ]:
assert A.T*u3 == sp.zeros(2, 1)
assert sp.Matrix.hstack(*A.columnspace(), *Ur.columnspace()).rank() == r
assert A.nullspace() == []
{
    'base_imagen': Ur.columnspace(),
    'base_espacio_fila': Vr.columnspace(),
    'base_nucleo_AT': [u3]
}

## 6. Pseudoinversa desde la SVD

En la SVD reducida basta invertir la diagonal positiva:

$$
A^+=V_r\Sigma_r^{-1}U_r^T.
$$

In [ ]:
Aplus = sp.simplify(Vr * Sigmar.inv() * Ur.T)
assert Aplus == sp.Matrix([[sp.Rational(3,25), sp.Rational(4,25), 0],
                           [0, 0, sp.Rational(1,2)]])
Aplus

## 7. Ecuaciones de Moore-Penrose

Las cuatro identidades caracterizan de manera única a la pseudoinversa.

In [ ]:
assert A*Aplus*A == A
assert Aplus*A*Aplus == Aplus
assert (A*Aplus).T == A*Aplus
assert (Aplus*A).T == Aplus*A
Pcol = A*Aplus
Prow = Aplus*A
assert Pcol**2 == Pcol and Prow**2 == Prow
Pcol, Prow

## 8. Sistema incompatible y mínimos cuadrados

Para una matriz con columnas dependientes, $A^+b$ selecciona entre todas las soluciones de mínimos cuadrados la de norma mínima.

In [ ]:
B = sp.Matrix([[1, 1, 0], [0, 0, 1], [1, 1, 1]])
b = sp.Matrix([1, 0, 2])
Bplus = B.pinv()
xplus = sp.simplify(Bplus*b)
residuo = sp.simplify(b - B*xplus)
assert B.T*residuo == sp.zeros(3, 1)
assert xplus.dot(sp.Matrix([1, -1, 0])) == 0
B.rank(), B.row_join(b).rank(), xplus, residuo

## 9. Comparación con una rutina numérica

Las columnas pueden cambiar de signo o rotar dentro de un subespacio singular repetido. Por ello se verifica la reconstrucción y no la igualdad literal entre columnas.

In [ ]:
A_np = np.array(A, dtype=float)
U_np, s_np, Vt_np = np.linalg.svd(A_np, full_matrices=True)
Sigma_np = np.zeros_like(A_np)
Sigma_np[:len(s_np), :len(s_np)] = np.diag(s_np)
assert np.allclose(U_np.T @ U_np, np.eye(m))
assert np.allclose(Vt_np @ Vt_np.T, np.eye(n))
assert np.allclose(U_np @ Sigma_np @ Vt_np, A_np)
assert np.allclose(np.linalg.pinv(A_np), np.array(Aplus, dtype=float))
s_np

## 10. Ejercicios para completar

1. Repite la construcción para una matriz $2\times3$ de rango dos y controla todos los tamaños.
2. Elige una matriz de rango uno y comprueba que su SVD reducida contiene un solo término.
3. Modifica $b$ en el ejemplo de mínimos cuadrados para que el sistema sea compatible y verifica que $A^+b$ sea la solución exacta de norma mínima.
4. Construye las proyecciones $BB^+$ y $B^+B$ y encuentra sus imágenes y núcleos.
5. Explica por qué comparar directamente los signos de las columnas producidas por dos rutinas SVD puede dar una falsa discrepancia.